# PYNQ-Z2 环境采集

先完成 HDMI close 并保存之前的测试记录，再运行下面唯一代码单元。
此测试本读取系统版本、设备树、已安装 Base 文件哈希和当前 PL IP 清单；不加载 Overlay。
完成后保存本文件，并从 Jupyter 文件列表下载生成的 board_environment.json，交给项目归档。
环境采集不代表新 Demo 通过，PCB 修订版仍需查看实体板丝印。


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

collector_source = '"""Read-only PYNQ-board inventory; never loads an overlay or reports a demo pass.\n\nCopy to the board after it arrives. Run with its Python, not Windows Python.\nPCB revision still requires checking the physical silkscreen.\n"""\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport pathlib\nimport platform\nimport sys\nfrom datetime import datetime, timezone\n\ndef read_text(path):\n    try:\n        return pathlib.Path(path).read_text(encoding="utf-8", errors="replace").strip().strip("\\x00")\n    except OSError:\n        return None\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output", type=pathlib.Path, default=pathlib.Path("board_environment.json"))\n    args = parser.parse_args()\n    if sys.platform != "linux":\n        raise SystemExit("This script must run on the Linux PYNQ board; it does not emulate board results on Windows")\n    import pynq\n    from pynq import PL\n    spec = importlib.util.find_spec("pynq.overlays.base")\n    base_files = []\n    if spec is not None and spec.origin:\n        folder = pathlib.Path(spec.origin).parent\n        for suffix in ("*.bit", "*.hwh", "*.tcl"):\n            for path in sorted(folder.glob(suffix)):\n                with path.open("rb") as source:\n                    digest = hashlib.sha256(source.read()).hexdigest()\n                base_files.append({"path": str(path), "bytes": path.stat().st_size, "sha256": digest})\n    current_ips = {}\n    for name, description in PL.ip_dict.items():\n        current_ips[name] = {key: description.get(key) for key in ("type", "phys_addr", "addr_range", "parameters")}\n    memory_lines = read_text("/proc/meminfo") or ""\n    report = {\n        "captured_utc": datetime.now(timezone.utc).isoformat(),\n        "scope": "Read-only metadata; no overlay load, GPIO toggle, HDMI or DMA test",\n        "python_version": sys.version, "pynq_version": getattr(pynq, "__version__", None),\n        "uname": platform.uname()._asdict(), "board_env": os.environ.get("BOARD"),\n        "device_tree_model": read_text("/proc/device-tree/model"),\n        "device_tree_compatible": read_text("/proc/device-tree/compatible"),\n        "os_release": read_text("/etc/os-release"),\n        "memtotal": next((line for line in memory_lines.splitlines() if line.startswith("MemTotal:")), None),\n        "pcb_revision": None, "pcb_revision_status": "Check physical silkscreen; not inferred from Board Files",\n        "installed_base_files": base_files, "currently_loaded_bitfile": getattr(PL, "bitfile_name", None),\n        "currently_loaded_ip_dict": current_ips,\n        "demo_status": "NOT_TESTED: environment collection does not prove a demo passed",\n    }\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str) + "\\n", encoding="utf-8")\n    print(str(args.output))\n\nif __name__ == "__main__":\n    main()\n'
if sys.platform != "linux":
    raise RuntimeError("Run this notebook on the PYNQ board.")
subprocess.run([sys.executable, "-c", collector_source,
                "--output", "board_environment.json"], check=True)
report = json.loads(Path("board_environment.json").read_text(encoding="utf-8"))
print("PYNQ:", report["pynq_version"])
print("Python:", report["python_version"].splitlines()[0])
print("Kernel:", report["uname"]["release"])
print("Board:", report["device_tree_model"])
print("Memory:", report["memtotal"])
print("Base files:")
for item in report["installed_base_files"]:
    print(item["path"], item["bytes"], item["sha256"])
print("IP count:", len(report["currently_loaded_ip_dict"]))
print("Saved: board_environment.json")
